# LAB-HW-07 — BRAM Neuron State / 第一次使用真实片上 RAM

**今天只新增一件事：** 把 Lesson 9 的抽象 `state[address]` 变成 KV260 上真实的片上 Block RAM（BRAM）。

前置：LSN-009、LAB-HW-06，以及已工作的 LAB-HW-06 PS/Linux runtime path。

**Project Trace:** RMD-013 · T-HW-007/T-HW-011

## 1. 先冻结一个很小的 state-memory contract

本 Lab 只冻结：

| 项目 | 数值 |
|---|---:|
| state word | 32 bits |
| word count | 1024 |
| logical capacity | 4096 bytes / 4 KiB |
| valid word index | 0..1023 |
| PS-visible base | `0xA0000000` |
| word `i` 的 byte offset | `4 * i` |

32-bit word 足够教学“可寻址 neuron state”，但**不**提前决定最终正式 neuron record 的布局。

本 Lab **不**声明 MOD-004 已完成。

## 2. 为什么不是“一大堆 register”？

register array 可以保存 state，但 FPGA 里还有专门的 memory block。Vivado 能从合适的 synchronous RAM RTL 推断这些资源。

本教学 store：

- write 在 clock edge 发生；
- read 是 synchronous；
- `(* ram_style = "block" *)` 表达 implementation intent；
- build 仍必须在结果中找到 RAMB18/RAMB36 primitive 才算资源 oracle 通过。

所以 attribute 只是“想这么实现”，**不是**证据。

## 3. KV260 上的真实路径

<svg xmlns="http://www.w3.org/2000/svg" width="1000" height="250" viewBox="0 0 1000 250" role="img" aria-label="KV260 LAB-HW-07 PS Linux to BRAM state path">
  <rect x="20" y="75" width="150" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="95" y="108" text-anchor="middle" font-size="14">Ubuntu / Python</text>
  <text x="95" y="132" text-anchor="middle" font-size="12">fixed /dev/mem</text>
  <rect x="205" y="75" width="155" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="282" y="108" text-anchor="middle" font-size="14">PS HPM0 FPD</text>
  <text x="282" y="132" text-anchor="middle" font-size="12">AXI master</text>
  <rect x="395" y="75" width="145" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="468" y="118" text-anchor="middle" font-size="14">SmartConnect</text>
  <rect x="575" y="75" width="175" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="662" y="106" text-anchor="middle" font-size="14">AXI BRAM Controller</text>
  <text x="662" y="132" text-anchor="middle" font-size="12">0xA0000000 / 4 KiB</text>
  <rect x="785" y="75" width="195" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="882" y="105" text-anchor="middle" font-size="14">neuron state BRAM</text>
  <text x="882" y="130" text-anchor="middle" font-size="12">1024 × 32-bit</text>
  <path d="M170 117 L205 117 M360 117 L395 117 M540 117 L575 117 M750 117 L785 117" stroke="#333" stroke-width="2"/>
  <polygon points="205,117 195,112 195,122" fill="#333"/><polygon points="395,117 385,112 385,122" fill="#333"/>
  <polygon points="575,117 565,112 565,122" fill="#333"/><polygon points="785,117 775,112 775,122" fill="#333"/>
</svg>

AMD/Xilinx K26 `base_gpio_bram` reference 也把 BRAM window 放在 `0xA0000000`。

AXI BRAM Controller 只是 platform adapter；这一章不要求你自己实现 AXI。

## 4. 最重要的 timing 概念：synchronous read

native memory port 是 clocked 的。

如果两个 clock edge 之间把 address 改成 7，output 不会因为 address wire 改了就立刻变成 state[7]；真正的 memory operation 发生在 active clock edge。

教学 RTL 还使用 **read-first**：如果同一个 clock edge 对同一地址同时 read/write，registered read output 得到旧值，而 memory 保存新值。

不要把它和 Python MMIO timing 混在一起。AXI/controller/software stack 中间隔着多个 cycle；Python test 证明的是 functional memory correctness，不是“host read 只要一个 cycle”。

## 5. 上板前先证明 native memory behavior

运行 open-source test：

```bash
iverilog -g2012 \
  -s kv260_neuron_state_store_tb \
  -o /tmp/lab07_state \
  boards/kv260/rtl/kv260_neuron_state_store.sv \
  boards/kv260/tb/kv260_neuron_state_store_tb.sv

vvp /tmp/lab07_state
```

最后应出现：

`PASS: kv260_neuron_state_store synchronous multi-address BRAM semantics`

这个 test 覆盖 multi-address、synchronous read、read-first 与 neighbor preservation。

## 6. Build 专用 LAB-HW-07 bitstream

development host 上执行：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-07-build.log \
  -source boards/kv260/scripts/build_lab07_bram_state.tcl
```

不能把“synthesis 成功”当作结束。写 bitstream 前必须满足：

- 没有 DRC error；
- setup slack 非负；
- hold slack 非负；
- `BRAM_PRIMITIVE_COUNT >= 1`。

保留 `utilization.rpt`；resource report 是本 Lab oracle 的一部分。

## 7. 不重启 Linux，直接配置 PL

LAB-HW-06 已经建立 development host / runtime host 的边界。

runtime host 上如有 active Kria app：

```bash
sudo xmutil unloadapp
```

然后 development host 上：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-07-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-07/kv260_bram_state.bit
```

之后不要 power-cycle；保持 PS/Linux 运行，只替换 PL configuration。

## 8. 没有硬件时先检查 host oracle

```bash
python boards/kv260/runtime/state_bram_mmio.py \
  --dry-run \
  --json-out /tmp/lab-hw-07-dry-run.json
```

dry-run 必须以 `STATUS=PASS` 结束。

它只检查 host-side address/value oracle，不证明 BRAM inference，也不证明真实 KV260 memory transaction。

## 9. 执行真实 multi-address state check

用已经可用的文件传输方式把 `state_bram_mmio.py` 放到 PS/Linux，然后：

```bash
sudo python3 /tmp/state_bram_mmio.py \
  --json-out /tmp/lab-hw-07-trace.json
```

checker 会向这些 index 写入不同值：

`0, 1, 7, 31, 255, 511, 1023`

随后：

1. 读回全部已写位置；
2. 改写选中的位置；
3. 再读全部 tracked location；
4. 证明没有改写的位置仍保持原 state。

实体 success 最后必须是 `STATUS=PASS`。

## 10. Failure class 要分开

host checker 区分：

- `TRANSPORT_DEVICE_MISSING`；
- `TRANSPORT_REQUIRES_ROOT`；
- `TRANSPORT_PERMISSION_OR_POLICY`；
- `TRANSPORT_MMAP_FAILED`；
- `STATE_READBACK_OR_ALIAS_MISMATCH`。

Vivado build 另外把 `NO_BLOCK_RAM_PRIMITIVE` 与 DRC/timing/build failure 分开。

如果 Ubuntu policy 阻止 `/dev/mem`，不要降低系统安全设置；保存 evidence，并保持 T-HW-007 blocked。

如果 `BRAM_PRIMITIVE_COUNT=0`，不能把“功能正常的 LUT memory”说成“BRAM 也算通过”。必须修正 implementation/resource mapping。

## 11. Expected Evidence / Save Evidence

保留：

- `lab-hw-07-build.log`；
- `timing_summary.rpt`、`utilization.rpt`、`drc.rpt`；
- `RAMB18_COUNT`、`RAMB36_COUNT`、`BRAM_PRIMITIVE_COUNT`；
- bitstream SHA-256；
- `lab-hw-07-program.log`；
- base `0xA0000000`、4 KiB window、1024 × 32-bit geometry；
- `state_bram_mmio.py` 的 SHA-256；
- 完整 runtime stdout；
- `lab-hw-07-trace.json`；
- Ubuntu/kernel identity、board/carrier revision、Git commit/date。

simulation PASS、dry-run PASS 或 resource report 单独都不能替代真实 T-HW-007 roundtrip evidence。

## 12. Human Check

请解释：

1. 为什么 `state[address]` 是 memory 问题，而不是 neuron arithmetic 问题？
2. 为什么 native BRAM address 改了，不代表 read data 异步立刻变化？
3. 同周期 read/write 时，read-first 是什么意思？
4. 为什么 `ram_style="block"` 本身不能证明真的用了 BRAM？
5. 为什么 `0xA0000000 + 4*i` 与 word index `i` 是两个不同量？
6. 为什么 Python/MMIO 能证明 addressable memory correctness，却不能证明 native one-cycle latency？
7. 为什么 LAB-HW-07 通过也不代表正式 MOD-004 已完成？

## 13. 官方依据

- AMD Vivado Design Suite User Guide: Synthesis（UG901），2026.1 — dedicated Block RAM 使用 synchronous read；Vivado 支持 RAM inference 与 `RAM_STYLE`。
- AMD AXI Block RAM Controller Product Guide（PG078）— AXI4-Lite mode 提供 32-bit single-beat BRAM access，并支持 single-port BRAM configuration。
- AMD/Xilinx `kria-base-hardware`，K26 `base_gpio_bram/scripts/config_bd.tcl` — PS `M_AXI_HPM0_FPD` → SmartConnect → AXI BRAM Controller reference path，以及 BRAM address `0xA0000000`。

仓库现在冻结的是教学 path；真实 T-HW-007 PASS 仍必须由实体 KV260 run 给出。